<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.6-remote-mcp-on-cloud-run/notebooks/exercises-8_6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.6: Remote MCP Servers on Cloud Run

*Module 8 · MCP, Tool Orchestration & Function Calling*

> Local MCP servers are great for development. Production needs HTTPS, authentication, auto-scaling, and zero cold-start. This lesson deploys an MCP server to Google Cloud Run with Streamable HTTP transport, secures it with IAM and API keys, and connects remote clients — including Claude Desktop over the internet.

`Cloud Run` · `Streamable HTTP` · `IAM Auth` · `Docker` · `75 min`

---

## About this notebook

This notebook contains the **5 runnable code examples** from the published lesson `lesson-8.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — HTTP Transport — From stdio to Streamable HTTP — `01_http_server.py`
2. Step 2 — Containerize — Dockerfile for Cloud Run — `Dockerfile`
3. Step 2 — Containerize — Dockerfile for Cloud Run — `requirements.txt`
4. Step 2 — Containerize — Dockerfile for Cloud Run — `02_deploy.sh`
5. Step 3 — Authentication — Secure Your MCP Endpoint — `03_auth_server.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q uvicorn


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export MCP_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("MCP_API_KEY", "YOUR_MCP_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · HTTP Transport — From stdio to Streamable HTTP

Replace stdin/stdout with an HTTP server that speaks MCP over Streamable HTTP.

**`01_http_server.py`** · _Block 1 of 5_

MCP SERVER WITH STREAMABLE HTTP TRANSPORT — pip install mcp starlette uvicorn


In [ ]:
# MCP SERVER WITH STREAMABLE HTTP TRANSPORT
# pip install mcp starlette uvicorn
from mcp.server import Server
from mcp.server.streamable_http import StreamableHTTPServerTransport
from mcp.types import Tool, TextContent
from starlette.applications import Starlette
from starlette.routing import Mount
import json, uvicorn

app = Server("netsetos-cloud")

COURSES = {
    "genai": {"name":"GenAI Complete","price":14999,"hours":146},
    "python": {"name":"Python Mastery","price":9999,"hours":80},
    "cloud": {"name":"GCP Cloud","price":11999,"hours":110},
}

@app.list_tools()
async def list_tools():
    return [
        Tool(name="search_courses", description="Search Netsetos courses by topic",
             inputSchema={"type":"object","properties":{"topic":{"type":"string"}},"required":["topic"]}),
        Tool(name="get_refund_policy", description="Check refund eligibility",
             inputSchema={"type":"object","properties":{"days":{"type":"integer"}},"required":["days"]}),
    ]

@app.call_tool()
async def call_tool(name, arguments):
    if name == "search_courses":
        course = COURSES.get(arguments["topic"].lower(), {"error":"Not found"})
        return [TextContent("text", json.dumps(course))]
    elif name == "get_refund_policy":
        d = arguments["days"]
        pct = "100%" if d<=7 else "50%" if d<=30 else "0%"
        return [TextContent("text", json.dumps({"refund":pct,"days":d}))]

# ── Streamable HTTP transport ──
transport = StreamableHTTPServerTransport(endpoint="/mcp")
starlette_app = Starlette(
    routes=[Mount("/mcp", app=transport.get_app())],
)

# Health check for Cloud Run
@starlette_app.route("/")
async def health(request):
    from starlette.responses import JSONResponse
    return JSONResponse({"status":"healthy","server":"netsetos-cloud"})

# Run locally: uvicorn 01_http_server:starlette_app --port 8080
if __name__ == "__main__":
    uvicorn.run(starlette_app, host="0.0.0.0", port=8080)


> **What just happened?** Same MCP server from 8.5, but now served over HTTP with StreamableHTTPServerTransport . The endpoint /mcp speaks MCP JSON-RPC over HTTP. The root / returns a health check for Cloud Run. Run locally with uvicorn , or deploy to Cloud Run — same code.


### Step 2 · Containerize — Dockerfile for Cloud Run

Package the server as a Docker image. Cloud Run runs containers.

**`Dockerfile`** · _Block 2 of 5_

Dockerfile for MCP Server on Cloud Run


In [ ]:
# Dockerfile for MCP Server on Cloud Run
FROM python:3.12-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy server code
COPY 01_http_server.py .

# Cloud Run uses PORT env variable
ENV PORT=8080
EXPOSE 8080

# Run with uvicorn
CMD ["uvicorn", "01_http_server:starlette_app", "--host", "0.0.0.0", "--port", "8080"]


**`requirements.txt`** · _Block 3 of 5_


In [ ]:
mcp>=1.0
starlette>=0.40
uvicorn>=0.30


**`02_deploy.sh`** · _Block 4 of 5_

!/bin/bash — Deploy MCP server to Cloud Run


In [ ]:
#!/bin/bash
# Deploy MCP server to Cloud Run

PROJECT_ID="your-gcp-project"
SERVICE="netsetos-mcp"
REGION="asia-south1"  # Mumbai (closest to Hyderabad)

# Build and push container
gcloud builds submit --tag gcr.io/$PROJECT_ID/$SERVICE

# Deploy to Cloud Run
gcloud run deploy $SERVICE \
  --image gcr.io/$PROJECT_ID/$SERVICE \
  --region $REGION \
  --platform managed \
  --allow-unauthenticated \
  --port 8080 \
  --memory 256Mi \
  --min-instances 0 \
  --max-instances 10 \
  --timeout 60s

# Get the URL
URL=$(gcloud run services describe $SERVICE --region $REGION --format "value(status.url)")
echo "MCP Server deployed at: $URL"
echo "MCP endpoint: $URL/mcp"
echo "Health check: $URL/"

# Test it
curl $URL/
# {"status":"healthy","server":"netsetos-cloud"}


> **What just happened?** Three commands: gcloud builds submit (build container), gcloud run deploy (deploy), curl (test). Your MCP server is now at https://netsetos-mcp-xxxxx.run.app/mcp . Auto-scaling from 0 to 10 instances. HTTPS built-in. 256MB memory. Cost: ~$0 when idle (scale to zero), ~$0.10/hr under load.


### Step 3 · Authentication — Secure Your MCP Endpoint

API key auth for simple cases, IAM for GCP-integrated systems.

**`03_auth_server.py`** · _Block 5 of 5_

MCP SERVER WITH API KEY AUTHENTICATION


In [ ]:
# MCP SERVER WITH API KEY AUTHENTICATION
from starlette.applications import Starlette
from starlette.middleware import Middleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse
import os

class APIKeyMiddleware(BaseHTTPMiddleware):
    """Validate API key on every request except health check."""
    async def dispatch(self, request, call_next):
        # Skip auth for health check
        if request.url.path == "/":
            return await call_next(request)

        # Check API key
        api_key = request.headers.get("X-API-Key") or request.query_params.get("api_key")
        expected = os.environ.get("MCP_API_KEY", "netsetos-secret-key")

        if api_key != expected:
            return JSONResponse({"error":"Invalid API key"}, status_code=401)

        return await call_next(request)

# Add middleware to Starlette app
# starlette_app = Starlette(
#     routes=[Mount("/mcp", app=transport.get_app())],
#     middleware=[Middleware(APIKeyMiddleware)]
# )

print("Authentication Options:\n")
print("  1. API Key (simple):")
print("     Header: X-API-Key: your-secret-key")
print("     Set MCP_API_KEY env var in Cloud Run\n")
print("  2. IAM (GCP-native):")
print("     gcloud run deploy ... --no-allow-unauthenticated")
print("     Client sends: Authorization: Bearer $(gcloud auth print-identity-token)\n")
print("  3. OAuth2 (production):")
print("     Use Google OAuth2 for user-level access")
print("     Integrate with Firebase Auth or Auth0")

# Cloud Run IAM deploy command
# gcloud run deploy netsetos-mcp \
#   --no-allow-unauthenticated \
#   --set-env-vars MCP_API_KEY=your-secret-key


> **What just happened?** Remote MCP servers on Cloud Run are accessible from any MCP client worldwide . The URL format is standard HTTPS. Each client has its own config format (covered in 8.5). OpenAI's hosted MCP approach is unique — their servers connect to your MCP server directly, no local client process needed.


---

## Wrap-up

You've walked through all 5 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
